# Rare-Class Benchmark: CNN Baselines (B0–B4) on HAM10000
**Research Goal:** Compare empirical risk minimization, classical augmentation, balanced sampling, cost-sensitive weighting, and focal loss on dermatofibroma (`df`, N=115).

**Zero-Leakage Protocol:** 5-fold StratifiedGroupKFold grouped strictly by `lesion_id`, repeated across 3 seeds (15 folds total).
**Self-Contained & Resumable:** Automatically checkpoints per-epoch, skips already-completed folds upon restart.


In [ ]:
# 1. Environment & Dependencies Setup
!pip install -q timm scikit-learn pandas matplotlib seaborn pyyaml tqdm


In [ ]:
# 2. Verify GPU Acceleration
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())


In [ ]:
# 3. Link or Mount HAM10000 Dataset
import os, glob
# On Kaggle, HAM10000 is typically mounted at /kaggle/input/skin-cancer-mnist-ham10000
kaggle_img_dirs = glob.glob('/kaggle/input/**/HAM10000_images_part_*', recursive=True)
print('Found Kaggle image parts:', kaggle_img_dirs)
os.makedirs('data/images', exist_ok=True)
for p in kaggle_img_dirs:
    !ln -sf {p}/* data/images/


In [ ]:
# 4. Run Leakage-Free Stratified Group Splits
!python scripts/make_splits.py


In [ ]:
# 5. Execute Baselines B0 through B4 Across All 15 Evaluations (5 folds x 3 seeds)
!python scripts/run_all.py --arms B0 B1 B2 B3 B4 --seeds 42 1337 2024 --folds 0 1 2 3 4


In [ ]:
# 6. Aggregate Results, Paired Statistics, and Summary Tables
from src.stats import aggregate_benchmark_runs
df_summary, df_runs = aggregate_benchmark_runs('results/logs/runs.csv', 'results/tables')
display(df_summary[['arm_id', 'n_evals', 'test_minority_recall_formatted', 'test_balanced_accuracy_formatted', 'test_macro_f1_formatted']])
